# EdgeFireUAV — Fire/Smoke Detector Training (Kaggle)

Runs entirely on Kaggle's free GPU. Before running:
1. Click **Add Input** (top right) → search for and add the dataset **`sayedgamal99/smoke-fire-detection-yolo`**.
2. Set **Settings → Accelerator → GPU T4 x2** (or P100) — training on CPU will be extremely slow.
3. Run All.

At the end you'll have a `fire_detector_outputs.zip` in the notebook's Output/working directory with both `best.pt` and `fire_detector.onnx` — download it from the notebook's Output panel, or via the Kaggle API, once the run finishes.

In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 657.4 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.4 MB/s eta 0:00:00


## Step 1 — Find the dataset's data.yaml

Kaggle dataset folder layouts vary, so this searches for it rather than assuming a fixed path. **Check the printed `names:` list against `fire_classes` in `edge/config/settings.yaml`** — if the order differs (e.g. `[smoke, fire]` instead of `[fire, smoke]`), update settings.yaml to match, since the class ID your model outputs has to line up with what settings.yaml expects.

In [2]:
import glob

# SET THIS if you know the exact path (copy it from the sidebar: hover over
# data.yaml under DATASETS -> Smoke-Fire-Detection-YOLO -> data -> click the
# copy-path icon). This skips auto-detection entirely and is the most
# reliable option -- auto-detection below can be fooled by an attached
# NOTEBOOK's old output, which sometimes mounts under a path that looks
# like a dataset path too.
MANUAL_DATA_YAML = '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data.yaml'  # e.g. '/kaggle/input/smoke-fire-detection-yolo/data/data.yaml'

if MANUAL_DATA_YAML:
    DATA_YAML = MANUAL_DATA_YAML
else:
    # Auto-detect, excluding anything that looks like it came from an
    # attached NOTEBOOK's output rather than an attached DATASET.
    all_yaml = glob.glob('/kaggle/input/**/*.yaml', recursive=True) + glob.glob('/kaggle/input/**/*.yml', recursive=True)
    candidates = [c for c in all_yaml if '/notebooks/' not in c and 'dataset' not in c.split('/')[-2].lower().replace('data', '')]
    print('Found candidate data.yaml files:')
    for c in all_yaml:
        print(' -', c)
    assert all_yaml, 'No data.yaml found under /kaggle/input at all -- did you Add Input the dataset?'
    print('\nIf the list above shows more than one file, or the wrong one gets picked below,')
    print('set MANUAL_DATA_YAML above instead of relying on auto-detection.')
    DATA_YAML = all_yaml[0]

print('\nUsing:', DATA_YAML)
print('\n--- contents ---')
with open(DATA_YAML) as f:
    print(f.read())



Using: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data.yaml

--- contents ---
path: /kaggle/working/D Fire Dataset  # dataset root dir
train: data/train/images  # train images (relative to 'path')
val: data/val/images  # val images (relative to 'path')
test: data/test/images  # test images (relative to 'path')

# Classes
names: ['smoke', 'fire']  # Replace with your actual class names

# Counts
nc: 2  # number of classes
train_count: 14122
val_count: 3099
test_count: 4306



## Step 1.5 — Auto-repair a broken `path:` in data.yaml

Some re-uploaded Kaggle datasets ship a `data.yaml` whose `path:` field is the
*original creator's own local folder* (e.g. `/kaggle/working/D Fire Dataset`),
which doesn't exist in your session. This cell ignores that broken `path:`,
tries a few real candidate roots near the yaml file itself, and writes a
corrected copy once it finds one where `train:`/`val:`/`test:` actually
resolve to real folders.

In [3]:
import os
import yaml

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

print('Declared classes:', data_cfg.get('names'))
print('(Make sure edge/config/settings.yaml fire_classes matches this exact order!)\n')

yaml_dir = os.path.dirname(os.path.abspath(DATA_YAML))
candidate_roots = [yaml_dir, os.path.dirname(yaml_dir), os.path.dirname(os.path.dirname(yaml_dir))]

def resolve(root, rel):
    return os.path.normpath(os.path.join(root, rel))

good_root = None
for root in candidate_roots:
    ok = True
    for split in ('train', 'val', 'test'):
        rel = data_cfg.get(split)
        if rel and not os.path.isdir(resolve(root, rel)):
            ok = False
            break
    if ok:
        good_root = root
        break

if good_root is None:
    print('Could NOT auto-resolve a working root. Diagnostics:')
    for root in candidate_roots:
        print(' candidate root:', root)
        for split in ('train', 'val', 'test'):
            rel = data_cfg.get(split)
            if rel:
                p = resolve(root, rel)
                print(f'   {split}: {p}  exists={os.path.isdir(p)}')
    raise SystemExit('Paste the diagnostics above back to Claude to fix the roots list.')

print('Resolved working dataset root:', good_root)
data_cfg['path'] = good_root
fixed_yaml_path = '/kaggle/working/data_fixed.yaml'
with open(fixed_yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f)

DATA_YAML = fixed_yaml_path
print('\nWrote corrected yaml to', fixed_yaml_path, '-- using this for training.\n')
print(open(fixed_yaml_path).read())

Declared classes: ['smoke', 'fire']
(Make sure edge/config/settings.yaml fire_classes matches this exact order!)

Resolved working dataset root: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo

Wrote corrected yaml to /kaggle/working/data_fixed.yaml -- using this for training.

names:
- smoke
- fire
nc: 2
path: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo
test: data/test/images
test_count: 4306
train: data/train/images
train_count: 14122
val: data/val/images
val_count: 3099



## Step 2 — Train

`yolov8n` (nano) is deliberately used — it's the size class that actually runs at usable FPS on a Jetson Nano later. `epochs=100` is a reasonable starting point; watch the printed mAP50 and reduce/increase as needed.

In [4]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    project='/kaggle/working/runs/fire_detector',
    name='train',
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_fixed.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      2.98G      1.699      1.913      1.517         15        640: 100% ━━━━━━━━━━━━ 882/882 6.6it/s 2:140.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 97/97 7.5it/s 12.9s0.1s
                   all       3094       3917      0.414      0.376      0.346      0.142

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      2.98G        1.8      1.974      1.606         10        640: 100% ━━━━━━━━━━━━ 882/882 6.7it/s 2:120.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 97/97 7.4it/s 13.0s0.1s
                   all       3094       3917      0.444      0.408      0.382      0.163

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      2.98G      1.831      1.991      1.639         14        640: 100% ━━━━━━━━━━━━ 882/882 6.8it/s 2:100.3ss
                 Class     Imag

In [5]:
metrics = model.val()
print(f'Validation mAP50: {metrics.box.map50:.3f}')

Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 178.9±189.9 MB/s, size: 129.4 KB)
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels... 3094 images, 1375 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 3099/3099 1.2Kit/s 2.6s0.1s
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection

## Step 3 — Export to ONNX

`opset=12` is chosen for compatibility with the older TensorRT (~8.2) that ships with a Jetson Nano's JetPack 4.6 — if `trtexec` on the Nano later complains about unsupported ops, try `opset=11` here and re-export.

In [6]:
import shutil

best_pt = '/kaggle/working/runs/fire_detector/train/weights/best.pt'

export_model = YOLO(best_pt)
exported_path = export_model.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.copy(str(exported_path), '/kaggle/working/fire_detector.onnx')
print('Wrote /kaggle/working/fire_detector.onnx')

Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/kaggle/working/runs/fire_detector/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 716ms
Prepared 2 packages in 973ms
Installed 2 packages in 12ms
 + onnxruntime==1.30.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 2.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 3.7

## Step 4 — Package everything for download

In [7]:
import shutil

shutil.copy(best_pt, '/kaggle/working/fire_detector_best.pt')

shutil.make_archive('/kaggle/working/fire_detector_outputs', 'zip', root_dir='/kaggle/working', base_dir='.',
                     ) if False else None

import zipfile
with zipfile.ZipFile('/kaggle/working/fire_detector_outputs.zip', 'w') as zf:
    zf.write('/kaggle/working/fire_detector.onnx', arcname='fire_detector.onnx')
    zf.write('/kaggle/working/fire_detector_best.pt', arcname='fire_detector_best.pt')

print('Done. Download fire_detector_outputs.zip from the notebook\'s Output panel (top right → Output).')
print('Only fire_detector.onnx needs to go to the Jetson Nano; keep the .pt for future retraining.')

Done. Download fire_detector_outputs.zip from the notebook's Output panel (top right → Output).
Only fire_detector.onnx needs to go to the Jetson Nano; keep the .pt for future retraining.


## Step 5 — Download the full runs/ folder (plots, curves, confusion matrix, etc.)

Everything under `/kaggle/working/runs/` — PR curves, confusion matrix, per-batch prediction images, `results.csv`, etc. — zipped up for download, separate from the `.onnx`/`.pt` package above since this is much larger and you likely only want it for inspection, not for the Jetson Nano.

In [8]:
import shutil

shutil.make_archive('/kaggle/working/runs_outputs', 'zip', root_dir='/kaggle/working', base_dir='runs')
print('Wrote /kaggle/working/runs_outputs.zip')

Wrote /kaggle/working/runs_outputs.zip
